### Data cleaning

In this notebook, I clean both datasets. I keep the original CSV files unchanged and save the cleaned data separately. Run the notebook with the project folder as the working directory so the relative file paths work.

##### Billboard dataset cleaning

In [36]:
import pandas as pd

# load the original Billboard data
billboard = pd.read_csv('/Users/apeksha/Desktop/Bootcamp/Final Project/data/raw/hot100.csv', low_memory=False)

print('Billboard shape:', billboard.shape)
billboard.head()

Billboard shape: (355758, 8)


,Date,Song,Artist,Rank,Last Week,Peak Position,Weeks in Charts,Image URL
0,1958-07-30,Poor Little Fool,Ricky Nelson,1,-,1,1,NaN
1,1958-07-30,Patricia,Perez Prado And His Orchestra,2,-,2,1,NaN
2,1958-07-30,Splish Splash,Bobby Darin,3,-,3,1,NaN
3,1958-07-30,Hard Headed Woman,Elvis Presley With The Jordanaires,4,-,4,1,https://charts-static.billboard.com/img/1958/0...
4,1958-07-30,When,Kalin Twins,5,-,5,1,NaN


In [37]:
# making a copy so raw data remain unchanged
billboard_clean = billboard.copy()

In [38]:
#rename the column names so it's easier to type in python
billboard_clean.rename(columns={
    'Date': 'date',
    'Song': 'song',
    'Artist': 'artist',
    'Rank': 'rank',
    'Last Week': 'last_week',
    'Peak Position': 'peak_position',
    'Weeks in Charts': 'weeks_in_charts',
    'Image URL': 'image_url'
}, inplace=True)

billboard_clean.columns.tolist()

['date',
 'song',
 'artist',
 'rank',
 'last_week',
 'peak_position',
 'weeks_in_charts',
 'image_url']

In [39]:
print('Date type before conversion:', billboard_clean['date'].dtype)

billboard_clean['date'] = pd.to_datetime(billboard_clean['date'], errors='coerce')

print('Date type after conversion:', billboard_clean['date'].dtype)
print('Missing dates:', billboard_clean['date'].isna().sum())

Date type before conversion: str
Date type after conversion: datetime64[us]
Missing dates: 0


##### Check missing values

last_week and weeks_in_charts columns contain "-" placeholders. So we turn it into missing value

In [40]:
numeric_columns = ['rank', 'last_week', 'peak_position', 'weeks_in_charts']

for column in numeric_columns:
    billboard_clean[column] = billboard_clean[column].replace('-', pd.NA)
    billboard_clean[column] = pd.to_numeric(billboard_clean[column], errors='coerce')

billboard_clean[numeric_columns].info()

<class 'pandas.DataFrame'>
RangeIndex: 355758 entries, 0 to 355757
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   rank             355758 non-null  int64  
 1   last_week        319367 non-null  float64
 2   peak_position    355758 non-null  int64  
 3   weeks_in_charts  355458 non-null  float64
dtypes: float64(2), int64(2)
memory usage: 10.9 MB


In [41]:
print('Missing values in each column:')
display(billboard_clean.isna().sum().to_frame('missing_values'))

Missing values in each column:


,missing_values
date,0
song,0
artist,0
rank,0
last_week,36391
peak_position,0
weeks_in_charts,300
image_url,124215


##### Check for duplicate rows

In [42]:
print('Exact duplicate rows:', billboard_clean.duplicated().sum())
print('Rows in cleaned data:', len(billboard_clean))

Exact duplicate rows: 0
Rows in cleaned data: 355758


##### Save the cleaned Billboard data


In [43]:
import os

os.makedirs('data/cleaned', exist_ok=True)
billboard_clean.to_csv('/Users/apeksha/Desktop/Bootcamp/Final Project/data/cleaned/billboard_cleaned.csv', index=False)
print('Cleaned Billboard data saved.')

Cleaned Billboard data saved.


### TikTok data cleaning

I will start by loading the TikTok CSV and checking its columns and data types. I will not change or filter the data in this step.

In [44]:
# load the original TikTok data
tiktok_trends = pd.read_csv('/Users/apeksha/Desktop/Bootcamp/Final Project/data/raw/trending_music.csv', low_memory=False)

print('TikTok shape:', tiktok_trends.shape)
tiktok_trends.head()

TikTok shape: (104886, 14)


,date,window_days,rank,song_id,clip_id,music_title,music_author,duration,rank_diff,rank_diff_type,rank_diff_label,is_commercial_music_library,is_promoted,country_code
0,2024-05-23,7,1,6844194558532651010,6844194559707088898,たぶん,YOASOBI,60,0.0,2,same,0,0,US
1,2024-05-23,7,2,7345183002882902018,7345176648368752641,Close,Cayo,60,NaN,4,new_to_top100,0,0,US
2,2024-05-23,7,3,6732292584862910465,6930510459246217217,Ghost Town,Kanye West,60,NaN,4,new_to_top100,0,0,US
3,2024-05-23,7,4,6995755961386272770,6995755962208356354,In My Mind,Lyn Lapid,60,NaN,4,new_to_top100,0,0,US
4,2024-05-23,7,5,6705086730946480129,6705026294012184578,Good Old Days,Macklemore & Kesha,60,NaN,4,new_to_top100,0,0,US


In [45]:
tiktok_trends.info()

<class 'pandas.DataFrame'>
RangeIndex: 104886 entries, 0 to 104885
Data columns (total 14 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   date                         104886 non-null  str    
 1   window_days                  104886 non-null  int64  
 2   rank                         104886 non-null  int64  
 3   song_id                      104886 non-null  int64  
 4   clip_id                      104886 non-null  int64  
 5   music_title                  104886 non-null  str    
 6   music_author                 104886 non-null  str    
 7   duration                     104886 non-null  int64  
 8   rank_diff                    77645 non-null   float64
 9   rank_diff_type               104886 non-null  int64  
 10  rank_diff_label              104886 non-null  str    
 11  is_commercial_music_library  104886 non-null  int64  
 12  is_promoted                  104886 non-null  int64  
 13  country_co

#### Look at missing `rank_diff` values
check whether the missing values are connected to a particular label. This helps us describe the pattern without guessing what the rank changes mean.

In [46]:
print('Missing rank_diff values:', tiktok_trends['rank_diff'].isna().sum())

print('Labels among rows with missing rank_diff:')
print(tiktok_trends.loc[tiktok_trends['rank_diff'].isna(), 'rank_diff_label'].value_counts(dropna=False))

pd.crosstab(tiktok_trends['rank_diff_label'], tiktok_trends['rank_diff'].isna(), margins=True)

Missing rank_diff values: 27241
Labels among rows with missing rank_diff:
rank_diff_label
new_to_top100    27241
Name: count, dtype: int64


rank_diff,False,True,All
rank_diff_label,,,
decrease,19337,0,19337
increase,38039,0,38039
new_to_top100,0,27241,27241
same,20269,0,20269
All,77645,27241,104886


In this file, all 27,241 missing `rank_diff` values have the label `new_to_top100`. This is a pattern in the data; I still need to check the dataset documentation before explaining what the labels mean.

##### Check repeated song records

In [47]:
# rows that share the same date, ranking window and song ID.
same_song_key = tiktok_trends.duplicated(
    subset=['date', 'window_days', 'song_id'],
    keep=False
)

print('Rows involved in repeated keys:', same_song_key.sum())

# display the records and keep them all for now. 
display(
    tiktok_trends.loc[same_song_key, [
        'date', 'window_days', 'song_id', 'music_title',
        'rank', 'clip_id', 'rank_diff', 'rank_diff_label'
    ]].sort_values(['date', 'song_id', 'rank'])
)

Rows involved in repeated keys: 4


,date,window_days,song_id,music_title,rank,clip_id,rank_diff,rank_diff_label
8964,2024-06-23,30,6886371996498675713,Mission Impossible (Funny Theme Remix),38,6886372003188574209,9.0,increase
8973,2024-06-23,30,6886371996498675713,Mission Impossible (Funny Theme Remix),47,6886372003188574209,7.0,increase
8963,2024-06-23,30,7378930449743710209,DEVIL IS A LIE,37,7378961993934407696,6.0,increase
8969,2024-06-23,30,7378930449743710209,DEVIL IS A LIE,43,7378961993934407696,NaN,new_to_top100


The four repeated-key rows are from June 23, 2024, in the 30-day window. Each pair has the same song and clip IDs but different ranks. I will keep them until we know why both ranks appear.

In [48]:
tiktok_clean = tiktok_trends.copy()

In [49]:
tiktok_clean['date'] = pd.to_datetime(tiktok_clean['date'], errors='coerce')

tiktok_clean['song_id'] = tiktok_clean['song_id'].astype('string')
tiktok_clean['clip_id'] = tiktok_clean['clip_id'].astype('string')

print('Date type:', tiktok_clean['date'].dtype)
print('Song ID type:', tiktok_clean['song_id'].dtype)
print('Clip ID type:', tiktok_clean['clip_id'].dtype)
print('Missing dates after conversion:', tiktok_clean['date'].isna().sum())

Date type: datetime64[us]
Song ID type: string
Clip ID type: string
Missing dates after conversion: 0


In [50]:
# check the rank_diff column if it contains whole numbers, missing values and so on

tiktok_clean['rank_diff'] = pd.to_numeric(
    tiktok_clean['rank_diff'], errors='coerce'
).astype('Int64')

print('rank_diff type:', tiktok_clean['rank_diff'].dtype)
print('Missing rank_diff values:', tiktok_clean['rank_diff'].isna().sum())
print('Exact duplicate rows:', tiktok_clean.duplicated().sum())

rank_diff type: Int64
Missing rank_diff values: 27241
Exact duplicate rows: 0


#### Create the seven-day TikTok subset
The project uses the seven-day window for its main comparison. so we save that subset separately and keep the other windows in the full cleaned file.

In [51]:
tiktok_7day = tiktok_clean.loc[tiktok_clean['window_days'] == 7].copy()

print('Seven-day rows:', len(tiktok_7day))
print('Distinct dates:', tiktok_7day['date'].nunique())
print('Rank range:', tiktok_7day['rank'].min(), 'to', tiktok_7day['rank'].max())
print('Rows per date, minimum and maximum:',
      tiktok_7day.groupby('date').size().min(),
      tiktok_7day.groupby('date').size().max())

Seven-day rows: 35602
Distinct dates: 358
Rank range: 1 to 100
Rows per date, minimum and maximum: 96 100


##### Save the cleaned TikTok data

In [52]:
os.makedirs('data/cleaned', exist_ok=True)

tiktok_clean.to_csv('/Users/apeksha/Desktop/Bootcamp/Final Project/data/cleaned/tiktok_cleaned.csv', index=False)
tiktok_7day.to_csv('/Users/apeksha/Desktop/Bootcamp/Final Project/data/cleaned/tiktok_7day_cleaned.csv', index=False)
print('Cleaned TikTok files saved.')

Cleaned TikTok files saved.
